# SQL Debug Agent：SFT → GRPO

先在菜单选择 **运行时 → 更改运行时类型 → T4 GPU**。按顺序运行每个单元格；先只做 5 步测试。

In [ ]:
from google.colab import files
uploaded = files.upload()  # 选择 artifacts/sql-debug-agent-colab.zip

In [ ]:
!unzip -q -o sql-debug-agent-colab.zip -d /content
%cd /content/sql-debug-agent-colab
!pip install -q -r cloud_grpo/requirements-colab.txt
!pip install -q -e .
print('安装完成。请在菜单中重启会话，再继续下一个单元格。')

## 重启会话后：SFT 5 步测试
成功标志是最后出现 `SFT Adapter 已保存`。

In [ ]:
%cd /content/sql-debug-agent-colab
import torch
assert torch.cuda.is_available(), '没有检测到 GPU，请先选择 T4 GPU'
print(torch.cuda.get_device_name(0))
!python cloud_grpo/train_sft.py --max-steps 5

## SFT 60 步
5 步成功后再运行。正式 Adapter 会保存到 Google Drive，避免 Colab 会话结束后丢失。

In [ ]:
%cd /content/sql-debug-agent-colab
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/sql-debug-agent-outputs
!python cloud_grpo/train_sft.py --max-steps 60 --output-dir /content/drive/MyDrive/sql-debug-agent-outputs/sft_adapter

## GRPO V2：5 步学习信号测试
使用 26 条真实失败难例，每题生成 4 个候选。成功标志是 `GRPO Adapter 已保存`；更重要的是末尾“有奖励差异”的步骤数不能一直为 0。

In [ ]:
%cd /content/sql-debug-agent-colab
!python cloud_grpo/train_grpo.py --max-steps 5 --dataset hard --num-generations 4 --temperature 1.2 --sft-adapter /content/drive/MyDrive/sql-debug-agent-outputs/sft_adapter --output-dir /content/drive/MyDrive/sql-debug-agent-outputs/grpo_v2_smoke

## 下载结果
先下载 5 步实验结果交回本地审计，再决定是否运行 40 步；不要只凭训练成功就宣布模型提升。

In [ ]:
%cd /content/sql-debug-agent-colab
!zip -qr sql-debug-agent-cloud-outputs.zip /content/drive/MyDrive/sql-debug-agent-outputs
from google.colab import files
files.download('sql-debug-agent-cloud-outputs.zip')